# ML Zoomcamp 2026 — Homework 6

Official: `materials/homework.md` (do not edit).
Data: `data/car_fuel_efficiency_2026.csv` (2026 pinned release).
Write-up: `ML_06_HW.md`.

Kernel: this folder `.venv`. Do not use `!pip` / `!wget`.
Submit: https://courses.datatalks.club/ml-zoomcamp-2026/homework/hw06


## 0) Imports


In [ ]:
# os.environ.setdefault sets a default only when the env var is missing.
# Keep Matplotlib/cache/temp on the E: drive (not C:) and pin numeric threads to 1.
import os
os.environ.setdefault("MPLCONFIGDIR", r"E:\IT_SPACES\AI\.cache\matplotlib")
os.environ.setdefault("XDG_CACHE_HOME", r"E:\IT_SPACES\AI\.cache")
os.environ.setdefault("TEMP", r"E:\IT_SPACES\AI\.cache\tmp")
os.environ.setdefault("TMP", r"E:\IT_SPACES\AI\.cache\tmp")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

# numpy handles numeric arrays; pandas handles table (DataFrame) data.
import numpy as np
import pandas as pd
# xgboost is the gradient-boosting library used in Q6.
import xgboost as xgb
# DictVectorizer turns row dicts into a numeric matrix (including one-hot encoding).
from sklearn.feature_extraction import DictVectorizer
# root_mean_squared_error computes RMSE between predictions and targets.
from sklearn.metrics import root_mean_squared_error
# train_test_split splits a table into train/val/test by ratio.
from sklearn.model_selection import train_test_split
# DecisionTreeRegressor builds a regression tree; export_text prints the rules as text.
from sklearn.tree import DecisionTreeRegressor, export_text
# RandomForestRegressor builds an ensemble of regression trees.
from sklearn.ensemble import RandomForestRegressor


## 1) Load data and split

Fill missing values with zeros. Split with the exact calls from the homework.


In [ ]:
# read_csv() loads a CSV file into a pandas DataFrame.
df = pd.read_csv("data/car_fuel_efficiency_2026.csv")
# fillna(0) replaces missing values with 0, matching the homework prep rules.
df = df.fillna(0)

# train_test_split() splits data by ratio: first hold out 20% as the test set.
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
# Split the remaining 80% 75:25 so overall train/val/test is 60/20/20.
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)

# .values pulls the fuel_efficiency_mpg target column as a NumPy array.
y_train = df_train.fuel_efficiency_mpg.values
y_val = df_val.fuel_efficiency_mpg.values

# del removes the label column so it does not leak into the feature matrix.
del df_train["fuel_efficiency_mpg"]
del df_val["fuel_efficiency_mpg"]

# to_dict(orient="records") builds a list of {column: value} dicts, one per row.
train_dicts = df_train.to_dict(orient="records")
val_dicts = df_val.to_dict(orient="records")

# DictVectorizer(sparse=True) one-hot encodes categories into a sparse numeric matrix.
dv = DictVectorizer(sparse=True)
# fit_transform learns category levels from train and builds X_train.
X_train = dv.fit_transform(train_dicts)
# transform applies the same column order to validation rows (no re-fitting).
X_val = dv.transform(val_dicts)

# .shape shows (rows, columns) so we can sanity-check the matrices.
df.shape, X_train.shape, X_val.shape


## Q1 — Decision tree, max_depth=1

Which original feature is used for splitting? Map a one-hot name such as `origin=USA` back to the original column.

- `vehicle_weight`
- `model_year`
- `origin`
- `fuel_type`

Answer:



In [ ]:
# DecisionTreeRegressor builds a decision tree for a numeric target (mpg).
# max_depth=1 allows only one split so we can see the root feature.
dt = DecisionTreeRegressor(max_depth=1)
# .fit learns the split rules from the train matrix and mpg labels.
dt.fit(X_train, y_train)
# export_text prints the learned rules; feature_names_ maps columns like origin=USA.
# Map a one-hot name back to the original column when picking the Q1 answer.
print(export_text(dt, feature_names=dv.feature_names_))


## Q2 — Random forest RMSE

`n_estimators=10`, `random_state=1`. Validation RMSE.

- 0.1837
- 1.837
- 18.37
- 183.7

Answer:



In [ ]:
# RandomForestRegressor is an ensemble of regression trees.
# n_estimators=10 uses 10 trees; random_state=1 makes the run reproducible.
rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
# .fit trains the forest on X_train and y_train.
rf.fit(X_train, y_train)
# .predict produces mpg predictions on the validation matrix.
y_pred = rf.predict(X_val)
# root_mean_squared_error computes validation RMSE to match the Q2 choices.
root_mean_squared_error(y_val, y_pred)


## Q3 — n_estimators

Try `[10, 50, 100, 150]`, `random_state=1`. Which value has the lowest validation RMSE (3 decimal places)?

- 10
- 50
- 100
- 150

Answer:



In [ ]:
# Try each n_estimators candidate and collect validation RMSE rounded to 3 decimals.
scores = []
for n in [10, 50, 100, 150]:
    # Rebuild a forest with this n_estimators; random_state=1 keeps results comparable.
    model = RandomForestRegressor(n_estimators=n, random_state=1, n_jobs=-1)
    model.fit(X_train, y_train)
    # Predict on val, compute RMSE, and round for the homework's 3-decimal comparison.
    rmse = root_mean_squared_error(y_val, model.predict(X_val))
    scores.append((n, round(rmse, 3)))
# scores holds (n_estimators, rounded RMSE) pairs; pick the lowest for Q3.
scores


## Q4 — max_depth by mean RMSE

`max_depth` in `[10, 15, 20, 25]`. For each, `n_estimators` in `[10, 50, 100, 150]`. `random_state=1`.

- 10
- 15
- 20
- 25

Answer:



In [ ]:
# For each max_depth, average validation RMSE over four n_estimators values.
depth_scores = []
for depth in [10, 15, 20, 25]:
    rmses = []
    for n in [10, 50, 100, 150]:
        # Train a forest at this depth and n_estimators.
        model = RandomForestRegressor(
            n_estimators=n, max_depth=depth, random_state=1, n_jobs=-1
        )
        model.fit(X_train, y_train)
        rmses.append(root_mean_squared_error(y_val, model.predict(X_val)))
    # np.mean averages the four RMSEs; round to 3 decimals for Q4.
    depth_scores.append((depth, round(float(np.mean(rmses)), 3)))
# Pick the max_depth with the best mean validation RMSE.
depth_scores


## Q5 — Most important feature

`n_estimators=10`, `max_depth=20`, `random_state=1`. Among `vehicle_weight`, `horsepower`, `acceleration`, `engine_displacement`.

Answer:



In [ ]:
# Fit a forest with the homework params for feature-importance ranking.
rf = RandomForestRegressor(
    n_estimators=10, max_depth=20, random_state=1, n_jobs=-1
)
rf.fit(X_train, y_train)
# feature_importances_ holds per-feature importance from split gain.
# pd.Series attaches DictVectorizer column names so we can read which feature won.
importances = pd.Series(rf.feature_importances_, index=dv.feature_names_)
# sort_values(ascending=False).head(10) shows the top features for Q5.
importances.sort_values(ascending=False).head(10)


## Q6 — XGBoost eta

100 rounds. Compare `eta` 0.3 and 0.1 on validation RMSE.

- 0.3
- 0.1
- Both give equal value

Answer:



In [ ]:
# feature_names_ is the column-name list from DictVectorizer (needed by DMatrix).
features = dv.feature_names_
# DMatrix is XGBoost's container for the matrix, labels, and feature names.
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=features)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=features)
# watchlist is (data, name) pairs evaluated during training.
watchlist = [(dtrain, "train"), (dval, "val")]

# xgb_params holds shared boosting settings; eta (learning rate) will vary below.
# objective="reg:squarederror" is the squared-error objective for mpg regression.
xgb_params = {
    "eta": 0.3,
    "max_depth": 6,
    "min_child_weight": 1,
    "objective": "reg:squarederror",
    "nthread": 8,
    "seed": 1,
    "verbosity": 1,
}

# Train once for each eta (0.3 and 0.1) and compare validation RMSE.
scores = {}
for eta in (0.3, 0.1):
    params = dict(xgb_params)
    params["eta"] = eta
    # num_boost_round=100 stacks 100 trees; verbose_eval=False hides per-round logs.
    model = xgb.train(params, dtrain, num_boost_round=100, evals=watchlist, verbose_eval=False)
    y_pred = model.predict(dval)
    scores[eta] = root_mean_squared_error(y_val, y_pred)
# scores maps eta -> val RMSE; pick the better eta (or a tie) for Q6.
scores
